# Module 05 — Lesson 08
# Merge and Join

---

> Real data almost never lives in one table. Customer names live in a `customers` table, their orders live in a separate `orders` table, and the only thing connecting them is a shared ID column. `.merge()` is how you combine two tables side by side using that shared key -- and *how* you combine them (which rows survive when a key doesn't match on both sides) is a decision with real consequences for your analysis.

We'll build two small example tables by hand first -- `customers` and `orders` -- because merge behavior is easiest to see with a handful of rows you can check by eye. Then we'll apply it to `tips.csv`, merging a `groupby()` summary back onto the original data.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Combine two DataFrames on a shared key column with `pd.merge()`
- Explain the difference between `how='inner'`, `'left'`, `'right'`, and `'outer'`, and when each row-survival rule matters
- Merge tables whose key columns have different names using `left_on` / `right_on`
- Merge a `groupby()` summary back onto the original detail rows
- Recognize when duplicate keys cause a merge to unexpectedly multiply rows

In [ ]:
import pandas as pd

customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "name": ["Aisha", "Ben", "Chen", "Diya"],
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104],
    "customer_id": [1, 2, 2, 5],   # customer_id 5 doesn't exist in `customers`
    "product": ["Laptop", "Mouse", "Keyboard", "Monitor"],
    "amount": [50000, 500, 1500, 8000],
})

print("customers:")
print(customers)
print()
print("orders:")
print(orders)

Notice the mismatch on purpose: **Chen (3)** and **Diya (4)** have no orders, and **order 104** belongs to a `customer_id` (5) that doesn't exist in `customers`. This is exactly the situation `how=` is built to handle.

---
## 1. `pd.merge()` — The Basic Inner Join

By default, `pd.merge()` does an **inner join**: keep only rows where the key exists on *both* sides. Anything unmatched on either side is dropped.

In [ ]:
inner = pd.merge(orders, customers, on="customer_id")
print(f"inner join: {inner.shape[0]} rows (started with {orders.shape[0]} orders, {customers.shape[0]} customers)")
print(inner)

Order 104 (customer_id 5) is gone -- no matching customer. Chen and Diya are gone too -- no matching orders. Only customer_ids present on *both* sides survive.

---
## 2. The Other Three Join Types

- **`how='left'`** -- keep every row from the *left* table, filling with `NaN` where the right table has no match
- **`how='right'`** -- keep every row from the *right* table, filling with `NaN` where the left table has no match
- **`how='outer'`** -- keep every row from *both* tables, filling `NaN` wherever there's no match on either side

In [ ]:
left = pd.merge(orders, customers, on="customer_id", how="left")
print("how='left' -- every order kept, even order 104 (unmatched customer_id):")
print(left)

In [ ]:
right = pd.merge(orders, customers, on="customer_id", how="right")
print("how='right' -- every customer kept, including Chen and Diya (no orders):")
print(right)

In [ ]:
outer = pd.merge(orders, customers, on="customer_id", how="outer")
print(f"how='outer' -- everything from both sides: {outer.shape[0]} rows")
print(outer)

---
## 3. Different Key Names: `left_on` / `right_on`

`on=` only works when both tables use the **same** column name for the key. When they don't, tell `merge()` which column plays that role on each side.

In [ ]:
customers_alt_name = customers.rename(columns={"customer_id": "cust_id"})
print("customers_alt_name has a differently-named key column:")
print(customers_alt_name)
print()

merged_alt = pd.merge(orders, customers_alt_name, left_on="customer_id", right_on="cust_id")
print("Merged using left_on / right_on:")
print(merged_alt)

---
## 4. A Real Use Case: Merging a `groupby()` Summary Back onto the Data

This is one of the most common real-world merges: compute a per-group summary, then merge it back onto every original row so each row can be compared against its group's average.

In [ ]:
tips = pd.read_csv("data/tips.csv")

avg_tip_by_day = tips.groupby("day")["tip"].mean().reset_index()
avg_tip_by_day = avg_tip_by_day.rename(columns={"tip": "avg_tip_for_day"})
print("Per-day average tip (the summary table):")
print(avg_tip_by_day)

In [ ]:
tips_with_avg = tips.merge(avg_tip_by_day, on="day")
tips_with_avg["above_day_average"] = tips_with_avg["tip"] > tips_with_avg["avg_tip_for_day"]

print(f"tips_with_avg.shape: {tips_with_avg.shape}  (same row count as tips -- every day has a match)")
print(tips_with_avg[["day", "tip", "avg_tip_for_day", "above_day_average"]].head(6))

---
## ⚠️ Common Mistakes

In [ ]:
# -- Mistake 1: Using the default inner join when you need to KEEP unmatched rows --

print("Mistake 1 — the default how='inner' silently drops unmatched rows:")
accidental_inner = pd.merge(orders, customers, on="customer_id")
print(f"  Default merge: {accidental_inner.shape[0]} rows -- order 104 vanished with NO warning")
print()
print("  Correct: decide how= deliberately. If you need every order even without")
print("  a matching customer, use how='left':")
print(f"  how='left': {pd.merge(orders, customers, on='customer_id', how='left').shape[0]} rows")

In [ ]:
# -- Mistake 2: Forgetting left_on/right_on when key columns have different names --

print("Mistake 2 — merge() can't find a shared column if the names don't match:")
try:
    pd.merge(orders, customers_alt_name, on="customer_id")
except KeyError as e:
    print(f"  Caught error: {e}")
print()
print("  Correct: use left_on / right_on to name each side's key explicitly:")
print(pd.merge(orders, customers_alt_name, left_on="customer_id", right_on="cust_id").shape)

In [ ]:
# -- Mistake 3: Duplicate keys silently multiplying rows -----------------------

print("Mistake 3 — if a key repeats on BOTH sides, merge() returns every combination:")
customers_with_dup = pd.concat([
    customers,
    pd.DataFrame({"customer_id": [2], "name": ["Ben (duplicate record)"]}),
], ignore_index=True)

print(f"  orders has customer_id=2 appearing {sum(orders['customer_id'] == 2)} time(s)")
print(f"  customers_with_dup has customer_id=2 appearing "
      f"{sum(customers_with_dup['customer_id'] == 2)} time(s)")

surprising = pd.merge(orders, customers_with_dup, on="customer_id")
print(f"  Result: {surprising.shape[0]} rows -- MORE than the {orders.shape[0]} orders we started with")
print(surprising[surprising["customer_id"] == 2])
print()
print("  Correct: always check for duplicate keys with .duplicated().sum() on the")
print("  join column BEFORE merging, if row count matters for your analysis.")

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Inner Join

Create two small DataFrames: `students` with columns `"student_id"` and `"name"` (3 students), and `grades` with columns `"student_id"` and `"grade"` (covering only 2 of those students). Merge them with the default inner join and print the result.

In [ ]:
# Your code here

### Exercise 2 — Left Join and Finding Unmatched Rows

Using `orders` and `customers` from the lesson, do a `how='left'` merge. Then use `.isna()` on the resulting `"name"` column to print only the order(s) that had no matching customer.

In [ ]:
# Your code here

### Exercise 3 — `left_on` / `right_on`

Using `orders` and `customers_alt_name` from the lesson, merge them with `left_on`/`right_on` and print only the `"order_id"`, `"product"`, and `"name"` columns of the result.

In [ ]:
# Your code here

### Exercise 4 — Merge a Summary Back onto the Data

Compute the average `"total_bill"` per `"time"` (Lunch/Dinner) using `groupby()`, then merge that summary back onto `tips`. Add a column `"bill_vs_time_avg"` that is `True` when a row's `total_bill` is above its time-of-day's average.

In [ ]:
# Your code here

### Exercise 5 — (Challenge) Detecting Duplicate Keys Before Merging

Before merging `orders` with `customers_with_dup` from the lesson, write code that checks whether `"customer_id"` has any duplicates in `customers_with_dup` using `.duplicated()`, and prints a warning message if so -- **before** running the merge that would multiply rows.

In [ ]:
# Your code here

---
## 📌 Key Takeaways

- **`pd.merge()` combines two tables on a shared key**, and `how=` controls which rows survive when a key doesn't match on both sides — `'inner'` (default, matches only), `'left'`, `'right'`, or `'outer'` (everything, gaps filled with `NaN`).

- **Use `left_on`/`right_on` when the key columns are named differently** in each table — `on=` only works when the names already match.

- **Duplicate keys on both sides of a merge multiply rows**, not just combine them — always sanity-check row counts (and `.duplicated()` on the key column) before trusting a merge's output size.

---
## What's Next?

**Lesson 09 — Apply (Custom Functions)** goes beyond merge and groupby's built-in aggregations, letting you run your own function across every row or column.